In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, RandomFlip, RandomRotation
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import os

# --- 1. DATA ACQUISITION & PREPROCESSING ---
print("Loading and Preprocessing Data...")
data_dir = '../data/train'
batch_size = 32
img_size = (224, 224)

# Load data and automatically split 80/20 for training and validation
train_dataset = image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=123,
    image_size=img_size, batch_size=batch_size
)

val_dataset = image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=123,
    image_size=img_size, batch_size=batch_size
)

class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Found {num_classes} classes: {class_names}")

# Data Augmentation (Optimization Technique 1)
data_augmentation = Sequential([
  RandomFlip('horizontal'),
  RandomRotation(0.2),
])

# Use pre-fetching for faster loading
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# --- 2. MODEL CREATION (Using Pre-trained Model) ---
print("\nBuilding Optimized Model...")
# Load MobileNetV2 without the top classification layer (Optimization Technique 2)
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Freeze the base model to retain learned features

# Build the custom classification head
inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x) # Built-in preprocessing
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs, outputs)

# Optimization Technique 3: Adam Optimizer
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Optimization Technique 4: Early Stopping (Prevents overfitting)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# --- 3. MODEL TRAINING ---
print("\nStarting Training...")
epochs = 10 # Set low because early stopping will catch it if it finishes sooner
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=epochs,
    callbacks=[early_stop]
)

# --- 4. MODEL EVALUATION & METRICS ---
print("\nEvaluating Model Metrics...")
# Get predictions for the validation set to calculate advanced metrics
y_true = np.concatenate([y for x, y in val_dataset], axis=0)
y_pred_probs = model.predict(val_dataset)
y_pred = np.argmax(y_pred_probs, axis=1)

# Calculate 4 distinct metrics for the rubric
loss, accuracy = model.evaluate(val_dataset, verbose=0)
print(f"\n1. Accuracy: {accuracy * 100:.2f}%")
print(f"2. Loss: {loss:.4f}")

# Print Precision, Recall, and F1-Score
print("\n3 & 4. Precision, Recall, and F1-Score:")
print(classification_report(y_true, y_pred, target_names=class_names))

# --- 5. EXPORT MODEL ---
os.makedirs('../models', exist_ok=True)
model_path = '../models/plant_disease_model.h5'
model.save(model_path)
print(f"\n✅ Phase 1 Complete! Model saved successfully to {model_path}")

Loading and Preprocessing Data...
Found 2152 files belonging to 3 classes.


Using 1722 files for training.


Found 2152 files belonging to 3 classes.


Using 430 files for validation.


Found 3 classes: ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']

Building Optimized Model...


...


13/14 [==========================>...] - ETA: 0s


14/14 [==============================] - ETA: 0s


14/14 [==============================] - 2s 116ms/step



1. Accuracy: 97.91%
2. Loss: 0.0661

3 & 4. Precision, Recall, and F1-Score:
                       precision    recall  f1-score   support

Potato___Early_blight       0.98      1.00      0.99       204
 Potato___Late_blight       0.99      0.96      0.98       197
     Potato___healthy       0.87      0.93      0.90        29

             accuracy                           0.98       430
            macro avg       0.95      0.97      0.96       430
         weighted avg       0.98      0.98      0.98       430


✅ Phase 1 Complete! Model saved successfully to ../models/plant_disease_model.h5


/Users/LENOVO/mlops_summative--1/venv/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
